# 歌词分词，词性标注

In [1]:
import json
import pandas as pd


# import thulac


from collections import Counter
from openai import OpenAI

In [2]:
# 可以选择是否加载
# jieba.load_userdict('data/mayday_dict_simple.txt')

In [3]:
import sys
sys.path.append('..')

# 分词，词频与词性分析

In [4]:
word_to_fix = {
    '阮': 'r',
    '袂': 'v',
    '春娇': 'n',
    '学会': 'v'
}

In [5]:
# def process_lyrics_with_jieba(text):
#     # 1. 词性标注与分词
#     # jieba.posseg 会同时返回词和词性
#     words_with_pos = pseg.cut(text)

    
#     # 2. 过滤无意义字符（标点、空格、单字符停用词）
#     filtered_data = []
#     for word, pos in words_with_pos:
#         # 排除标点符号（x表示标点）及空白字符
#         if pos != 'x' and len(word.strip()) > 0:
#             if word in word_to_fix:
#                 filtered_data.append((word, word_to_fix[word]))
#             else:
#                 filtered_data.append((word, pos))
    
#     # 3. 统计词频
#     word_counts = Counter([item[0] for item in filtered_data])
    
#     # 4. 汇总信息 (词, 词性, 频数)
#     # 我们以词为 Key，存储词性
#     word_pos_map = {word: pos for word, pos in filtered_data}
    
#     # 排序：按词频从高到低
#     sorted_results = []
#     for word, count in word_counts.most_common():
#         sorted_results.append({
#             "word": word,
#             "pos": word_pos_map[word],
#             "freq": count # 词频
#         })
    
#     return sorted_results

In [6]:
import re
from hanlp_restful import HanLPClient
HanLP = HanLPClient('https://www.hanlp.com/hanlp/v21/redirect', auth="699691e7eaf61a3aca90d7b8", language='zh')

def is_chinese_word(word):
    """
    判断是否为纯中文词
    """
    return 1 if re.fullmatch(r'[\u4e00-\u9fff]+', word) else 0


def process_lyrics_with_hanlp_multi_pos(text, word_to_fix=None):
    if not text:
        return []
    
    # 调用 HanLP
    result = HanLP.parse(text, tasks='pos/pku')
    
    sentences = result['tok/fine']
    pos_sentences = result['pos/pku']
    
    # 统计 (word, pos) -> freq
    word_pos_counter = Counter()
    
    for words, pos_tags in zip(sentences, pos_sentences):
        for word, tag in zip(words, pos_tags):
            
            word = word.strip()
            
            # 过滤标点
            if tag == 'w' or not word:
                continue
            
            # 词性修正
            if word_to_fix and word in word_to_fix:
                tag = word_to_fix[word]
            
            word_pos_counter[(word, tag)] += 1
    
    # 构建结果列表
    results = []
    for (word, pos), freq in word_pos_counter.items():
        results.append({
            "word": word,
            "pos": pos,
            "freq": freq,
            "is_chinese": is_chinese_word(word)
        })
    
    # 按词频排序
    results.sort(key=lambda x: x["freq"], reverse=True)
    
    return results


In [7]:
# thu = thulac.thulac(seg_only=False, filt=True) 

# def process_lyrics_with_thulac(text, word_to_fix=None):
#     if not text:
#         return []
    
#     # 2. 执行分词与词性标注
#     # 返回格式为 [[word, pos], [word, pos], ...]
#     words_with_pos = thu.cut(text)
    
#     # 3. 过滤无意义字符与词性修正
#     # thulac 的标点词性通常是 'w'
#     filtered_data = []
#     for word, pos in words_with_pos:
#         word = word.strip()
#         # 排除标点符号、空白字符
#         if pos != 'w' and len(word) > 0:
#             # 逻辑修正：word_to_fix 通常是修正词性
#             if word_to_fix and word in word_to_fix:
#                 filtered_data.append((word, word_to_fix[word]))
#             else:
#                 filtered_data.append((word, pos))
    
#     # 4. 统计词频
#     word_counts = Counter([item[0] for item in filtered_data])
    
#     # 5. 汇总信息
#     # 建立 word -> pos 映射
#     word_pos_map = {word: pos for word, pos in filtered_data}
    
#     sorted_results = []
#     for word, count in word_counts.most_common():
#         sorted_results.append({
#             "word": word,
#             "pos": word_pos_map[word],
#             "freq": count
#         })
    
#     return sorted_results

In [8]:
def lyric_words_process(path_prefix, word_to_fix=None):
    lyric_file_path = path_prefix + 'cleared_lyric_data.json'
    # 读取歌词文件
    with open(lyric_file_path, 'r') as f:
        lyric_data = json.load(f)
    lyric_words_dict = {}
    for i in lyric_data:
        if i:
            # lyric_words_dict[i['song_id']] = process_lyrics_with_jieba(
            #     i['lyrics_text'])
            # lyric_words_dict[i['song_id']] = process_lyrics_with_thulac(
            #     i['lyrics_text'], word_to_fix=word_to_fix)
            lyric_words_dict[i['song_id']] = process_lyrics_with_hanlp_multi_pos(
                i['lyrics_text'], word_to_fix=word_to_fix)
    rows = []
    for song_id, word_list in lyric_words_dict.items():
        for item in word_list:
            # 创建新字典，保留原始数据并加入歌曲ID列
            new_row = {
                'song_id': song_id,
                'word': item['word'],
                'pos': item['pos'],
                'freq': item['freq']
            }
            rows.append(new_row)

    # 3. 转换为 DataFrame
    df_word = pd.DataFrame(rows)
    return df_word

In [26]:
def words_data_merge(df_word, df_songs):
    # 合并
    # 1. 确保 df_word 的 song_id 是字符串
    df_word = df_word.copy()
    df_songs = df_songs.copy()
    df_word['song_id'] = df_word['song_id'].astype(str)
    df_word['is_chinese'] = df_word['word'].apply(is_chinese_word)

    # 2. 确保 df_unique 的 song_id 是字符串（并去掉可能存在的空格）
    df_songs['song_id'] = df_songs['song_id'].astype(str).str.strip()

    # 3. 执行合并
    df_merged = df_word.merge(df_songs, on='song_id', how='left')

    # 4. 删除空值
    # df_merged = df_merged.dropna()

    return df_merged

# main

In [10]:
# file_path_prefix = "data/jaychou/"
file_path_prefix = "data/mayday/"
# file_path_prefix = "data/liuyuning/"
# file_path_prefix = "data/newyear/"

In [11]:
# 歌曲数据
df_songs = pd.read_csv(file_path_prefix + "cleared_song_data.csv")
df_songs['song_name_unique'] = df_songs['song_name_unique'].astype(str)
# 清洗song_name_unique，删除空格，将中文括号转为英文括号
df_songs['song_name_unique'] = df_songs['song_name_unique'].str.replace(' ', '')
df_songs['song_name_unique'] = df_songs['song_name_unique'].str.replace('（', '(').str.replace('）', ')')
df_songs

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_mid,duration,publish_time,song_name_unique,album_id,publish_date
0,107709592,0022QuVR1LcRHN,后来的我们,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74,000Sp0Bz4JXH0o,自传,002fRO0N4FftzY,346,1469030400,后来的我们,1393445,2016-07-21
1,447807,002M8hNI2QgtRY,突然好想你,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74,000Sp0Bz4JXH0o,后青春期的诗,0020I7sO0ayXhN,265,1224691200,突然好想你,36459,2008-10-23
2,5131923,003lhef916qYN2,步步,《步步惊情》电视剧主题曲,五月天,74,000Sp0Bz4JXH0o,步步 自选作品辑,0006MmDz4Hl2Ud,273,1388332800,步步,451706,2013-12-30
3,4830286,0033P66R0qEtlT,知足,《后来的我们》电影插曲,五月天,74,000Sp0Bz4JXH0o,知足 最真杰作选,003PIMo40rxcAn,256,1124985600,知足,96397,2005-08-26
4,106528423,000aHM1h2bD5Kb,派对动物,NaN,五月天,74,000Sp0Bz4JXH0o,自传,002fRO0N4FftzY,249,1469030400,派对动物,1393445,2016-07-21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132,4996096,003Xy9E32vvMLe,入阵曲,《兰陵王》电视剧主题曲,五月天,74,000Sp0Bz4JXH0o,步步 自选作品辑,002adz882rV5uh,209,1377792000,入阵曲,451706,2013-12-30
133,4932058,003PaRAX3j5wJk,生命有一种绝对,NaN,五月天,74,000Sp0Bz4JXH0o,时光机,0015r2I31enfaR,239,1038672000,生命有一种绝对,96353,2003-11-07
134,4830242,000PoJAV4NPMzW,温柔 (还你自由版),NaN,五月天,74,000Sp0Bz4JXH0o,知足 最真杰作选,001ntd0y01uQ4g,426,1088611200,温柔(还你自由版),96397,2005-08-26
135,4834459,002nqyCb1bUnk6,Enrich Your Life,NaN,五月天,74,000Sp0Bz4JXH0o,神的孩子都在跳舞,0006oAnx03zXUC,166,1096560000,EnrichYourLife,96368,2004-11-01


In [13]:
df_songs = df_songs.drop_duplicates(subset='song_name_unique', keep='first')
df_songs

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_mid,duration,publish_time,song_name_unique,album_id,publish_date
0,107709592,0022QuVR1LcRHN,后来的我们,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74,000Sp0Bz4JXH0o,自传,002fRO0N4FftzY,346,1469030400,后来的我们,1393445,2016-07-21
1,447807,002M8hNI2QgtRY,突然好想你,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74,000Sp0Bz4JXH0o,后青春期的诗,0020I7sO0ayXhN,265,1224691200,突然好想你,36459,2008-10-23
2,5131923,003lhef916qYN2,步步,《步步惊情》电视剧主题曲,五月天,74,000Sp0Bz4JXH0o,步步 自选作品辑,0006MmDz4Hl2Ud,273,1388332800,步步,451706,2013-12-30
3,4830286,0033P66R0qEtlT,知足,《后来的我们》电影插曲,五月天,74,000Sp0Bz4JXH0o,知足 最真杰作选,003PIMo40rxcAn,256,1124985600,知足,96397,2005-08-26
4,106528423,000aHM1h2bD5Kb,派对动物,NaN,五月天,74,000Sp0Bz4JXH0o,自传,002fRO0N4FftzY,249,1469030400,派对动物,1393445,2016-07-21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132,4996096,003Xy9E32vvMLe,入阵曲,《兰陵王》电视剧主题曲,五月天,74,000Sp0Bz4JXH0o,步步 自选作品辑,002adz882rV5uh,209,1377792000,入阵曲,451706,2013-12-30
133,4932058,003PaRAX3j5wJk,生命有一种绝对,NaN,五月天,74,000Sp0Bz4JXH0o,时光机,0015r2I31enfaR,239,1038672000,生命有一种绝对,96353,2003-11-07
134,4830242,000PoJAV4NPMzW,温柔 (还你自由版),NaN,五月天,74,000Sp0Bz4JXH0o,知足 最真杰作选,001ntd0y01uQ4g,426,1088611200,温柔(还你自由版),96397,2005-08-26
135,4834459,002nqyCb1bUnk6,Enrich Your Life,NaN,五月天,74,000Sp0Bz4JXH0o,神的孩子都在跳舞,0006oAnx03zXUC,166,1096560000,EnrichYourLife,96368,2004-11-01


In [14]:
# 更新cleared_song_data.csv
df_songs.to_csv(file_path_prefix + 'cleared_song_data.csv', index=False)

In [ ]:
# 五月天需要使用word_to_fix
# if file_path_prefix == "data/mayday/":
#     df_word = lyric_words_process(file_path_prefix, word_to_fix)
# else:
#     df_word = lyric_words_process(file_path_prefix, word_to_fix=None)

In [15]:
# hanlp暂时不需要word_to_fix
df_word = lyric_words_process(file_path_prefix, word_to_fix=None)
df_word

,song_id,word,pos,freq
0,107709592,的,u,20
1,107709592,了,y,15
2,107709592,后来,t,15
3,107709592,着,u,14
4,107709592,你,r,10
...,...,...,...,...
13714,519403016,请,v,1
13715,519403016,忘,v,1
13716,519403016,了,u,1
13717,519403016,还,v,1


In [27]:
df_merged = words_data_merge(df_word, df_songs)
df_merged = df_merged.dropna(subset='song_name')
df_merged

,song_id,word,pos,freq,is_chinese,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_mid,duration,publish_time,song_name_unique,album_id,publish_date
0,107709592,的,u,20,1,0022QuVR1LcRHN,后来的我们,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74.0,000Sp0Bz4JXH0o,自传,002fRO0N4FftzY,346.0,1.469030e+09,后来的我们,1393445.0,2016-07-21
1,107709592,了,y,15,1,0022QuVR1LcRHN,后来的我们,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74.0,000Sp0Bz4JXH0o,自传,002fRO0N4FftzY,346.0,1.469030e+09,后来的我们,1393445.0,2016-07-21
2,107709592,后来,t,15,1,0022QuVR1LcRHN,后来的我们,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74.0,000Sp0Bz4JXH0o,自传,002fRO0N4FftzY,346.0,1.469030e+09,后来的我们,1393445.0,2016-07-21
3,107709592,着,u,14,1,0022QuVR1LcRHN,后来的我们,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74.0,000Sp0Bz4JXH0o,自传,002fRO0N4FftzY,346.0,1.469030e+09,后来的我们,1393445.0,2016-07-21
4,107709592,你,r,10,1,0022QuVR1LcRHN,后来的我们,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74.0,000Sp0Bz4JXH0o,自传,002fRO0N4FftzY,346.0,1.469030e+09,后来的我们,1393445.0,2016-07-21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13714,519403016,请,v,1,1,000B69Qg0S8WUF,我不愿让你一个人,《今夜一起为爱鼓掌》电视剧插曲,五月天,74.0,000Sp0Bz4JXH0o,第二人生,001Jhk1t0SC1FZ,265.0,1.727626e+09,我不愿让你一个人,90142.0,2011-12-16
13715,519403016,忘,v,1,1,000B69Qg0S8WUF,我不愿让你一个人,《今夜一起为爱鼓掌》电视剧插曲,五月天,74.0,000Sp0Bz4JXH0o,第二人生,001Jhk1t0SC1FZ,265.0,1.727626e+09,我不愿让你一个人,90142.0,2011-12-16
13716,519403016,了,u,1,1,000B69Qg0S8WUF,我不愿让你一个人,《今夜一起为爱鼓掌》电视剧插曲,五月天,74.0,000Sp0Bz4JXH0o,第二人生,001Jhk1t0SC1FZ,265.0,1.727626e+09,我不愿让你一个人,90142.0,2011-12-16
13717,519403016,还,v,1,1,000B69Qg0S8WUF,我不愿让你一个人,《今夜一起为爱鼓掌》电视剧插曲,五月天,74.0,000Sp0Bz4JXH0o,第二人生,001Jhk1t0SC1FZ,265.0,1.727626e+09,我不愿让你一个人,90142.0,2011-12-16


In [29]:
df_merged_chn = df_merged[df_merged['is_chinese'] == 1]
df_merged_chn

,song_id,word,pos,freq,is_chinese,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_mid,duration,publish_time,song_name_unique,album_id,publish_date
0,107709592,的,u,20,1,0022QuVR1LcRHN,后来的我们,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74.0,000Sp0Bz4JXH0o,自传,002fRO0N4FftzY,346.0,1.469030e+09,后来的我们,1393445.0,2016-07-21
1,107709592,了,y,15,1,0022QuVR1LcRHN,后来的我们,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74.0,000Sp0Bz4JXH0o,自传,002fRO0N4FftzY,346.0,1.469030e+09,后来的我们,1393445.0,2016-07-21
2,107709592,后来,t,15,1,0022QuVR1LcRHN,后来的我们,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74.0,000Sp0Bz4JXH0o,自传,002fRO0N4FftzY,346.0,1.469030e+09,后来的我们,1393445.0,2016-07-21
3,107709592,着,u,14,1,0022QuVR1LcRHN,后来的我们,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74.0,000Sp0Bz4JXH0o,自传,002fRO0N4FftzY,346.0,1.469030e+09,后来的我们,1393445.0,2016-07-21
4,107709592,你,r,10,1,0022QuVR1LcRHN,后来的我们,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74.0,000Sp0Bz4JXH0o,自传,002fRO0N4FftzY,346.0,1.469030e+09,后来的我们,1393445.0,2016-07-21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13714,519403016,请,v,1,1,000B69Qg0S8WUF,我不愿让你一个人,《今夜一起为爱鼓掌》电视剧插曲,五月天,74.0,000Sp0Bz4JXH0o,第二人生,001Jhk1t0SC1FZ,265.0,1.727626e+09,我不愿让你一个人,90142.0,2011-12-16
13715,519403016,忘,v,1,1,000B69Qg0S8WUF,我不愿让你一个人,《今夜一起为爱鼓掌》电视剧插曲,五月天,74.0,000Sp0Bz4JXH0o,第二人生,001Jhk1t0SC1FZ,265.0,1.727626e+09,我不愿让你一个人,90142.0,2011-12-16
13716,519403016,了,u,1,1,000B69Qg0S8WUF,我不愿让你一个人,《今夜一起为爱鼓掌》电视剧插曲,五月天,74.0,000Sp0Bz4JXH0o,第二人生,001Jhk1t0SC1FZ,265.0,1.727626e+09,我不愿让你一个人,90142.0,2011-12-16
13717,519403016,还,v,1,1,000B69Qg0S8WUF,我不愿让你一个人,《今夜一起为爱鼓掌》电视剧插曲,五月天,74.0,000Sp0Bz4JXH0o,第二人生,001Jhk1t0SC1FZ,265.0,1.727626e+09,我不愿让你一个人,90142.0,2011-12-16


In [30]:
df_merged_chn.to_csv(file_path_prefix + "cleared_words_data.csv", index=False)

In [19]:
df_merged_chn['pos'].unique()

array(['u', 'y', 't', 'r', 'd', 'v', 'n', 'a', 'c', 'f', 'm', 'q', 'p',
       'an', 'z', 'ad', 'vn', 'Ng', 's', 'l', 'b', 'i', 'ns', 'o', 'nz',
       'nr', 'Vg', 'Ag', 'Tg', 'nt', 'e', 'k', 'vd', 'j', 'Mg', 'Rg'],
      dtype=object)

# 测试